# 🚀 micro1 Python Practice — Task Assignment Problems
Work through each section, run the code, and try the **challenge** at the end!

---

## 1️⃣ Minimize Maximum Load (Heap-based)
**Problem:** Given a list of tasks (each with a numeric load), assign them to `k` teams such that the **maximum load on any team is minimized**.

**Approach:** Use a min-heap. Always assign the next task to the team with the least current load.

In [ ]:
import heapq

def minimize_max_load(tasks, num_teams):
    """
    Assigns tasks to teams to minimize the maximum load.
    Returns the minimum possible maximum load.
    """
    heap = [0] * num_teams
    heapq.heapify(heap)

    for task in sorted(tasks, reverse=True):  # assign biggest tasks first
        least_loaded = heapq.heappop(heap)
        heapq.heappush(heap, least_loaded + task)

    print(f"Tasks: {tasks}")
    print(f"Teams: {num_teams}")
    print(f"Team loads: {sorted(heap)}")
    print(f"Minimum possible max load: {max(heap)}")
    return max(heap)

# --- Test ---
tasks = [3, 1, 4, 2, 8, 5]
minimize_max_load(tasks, num_teams=3)

### 🧪 Try it yourself
Modify `tasks` and `num_teams` and re-run!

In [ ]:
# ✏️ Your turn — change these values
my_tasks = [10, 20, 30, 40, 50]
my_teams = 2
minimize_max_load(my_tasks, my_teams)

---
## 2️⃣ Assign Tasks and Return Team Map
**Problem:** Same as above, but also return **which tasks go to which team**.

In [ ]:
import heapq

def assign_tasks_map(tasks, k):
    """
    Returns a dict mapping team index -> list of assigned tasks.
    Greedily assigns each task to the team with the current minimum load.
    """
    # heap stores (current_load, team_index)
    heap = [(0, i) for i in range(k)]
    heapq.heapify(heap)

    assignment = {i: [] for i in range(k)}

    for task in sorted(tasks, reverse=True):
        load, team = heapq.heappop(heap)
        assignment[team].append(task)
        heapq.heappush(heap, (load + task, team))

    print(f"Tasks: {tasks} | Teams: {k}\n")
    for team, t in assignment.items():
        print(f"  Team {team}: tasks={t}, total load={sum(t)}")

    return assignment

# --- Test ---
tasks = [5, 3, 8, 1, 2, 7]
assign_tasks_map(tasks, k=2)

---
## 3️⃣ Task Scheduling — Earliest Deadline First
**Problem:** Each task has a name, duration, and deadline. Schedule tasks to **maximize the number completed on time**.

In [ ]:
def schedule_tasks(tasks):
    """
    tasks: list of (name, duration, deadline)
    Returns list of tasks completed on time, sorted by deadline.
    """
    tasks_sorted = sorted(tasks, key=lambda x: x[2])  # sort by deadline

    time = 0
    completed = []
    skipped = []

    for name, duration, deadline in tasks_sorted:
        if time + duration <= deadline:
            time += duration
            completed.append((name, deadline))
        else:
            skipped.append((name, deadline))

    print("✅ Completed on time:", [t[0] for t in completed])
    print("❌ Missed deadline: ", [t[0] for t in skipped])
    print(f"Total time used: {time}")
    return completed

# --- Test ---
tasks = [
    ("Task A", 2, 5),
    ("Task B", 3, 4),
    ("Task C", 1, 2),
    ("Task D", 4, 10),
    ("Task E", 6, 7),
]
schedule_tasks(tasks)

---
## 4️⃣ Round Robin Assignment
**Problem:** Distribute tasks evenly across teams in a rotating (round-robin) fashion.

In [ ]:
def round_robin_assign(tasks, k):
    """
    Assigns tasks to k teams in round-robin order.
    Each team gets every k-th task.
    """
    teams = {i: [] for i in range(k)}

    for i, task in enumerate(tasks):
        teams[i % k].append(task)

    print(f"Tasks: {tasks} | Teams: {k}\n")
    for team, t in teams.items():
        print(f"  Team {team}: {t}")

    return teams

# --- Test ---
tasks = ["T1", "T2", "T3", "T4", "T5", "T6", "T7"]
round_robin_assign(tasks, k=3)

---
## 5️⃣ Optimal Partition — Binary Search + Greedy
**Problem:** Split a list of tasks into `k` groups (teams) such that the **maximum group sum is minimized**. This is the most advanced pattern.

**Key Insight:** Binary search on the answer (max load), and use greedy to check if a given max load is feasible.

In [ ]:
def can_distribute(tasks, k, max_load):
    """
    Greedy check: can we split tasks into at most k groups
    where each group sum <= max_load?
    """
    teams_needed = 1
    current_load = 0

    for task in tasks:
        if task > max_load:
            return False  # single task exceeds max_load
        if current_load + task > max_load:
            teams_needed += 1
            current_load = task
        else:
            current_load += task

    return teams_needed <= k


def optimal_partition(tasks, k):
    """
    Binary search to find the minimum possible maximum load
    when splitting tasks into k groups.
    """
    tasks = sorted(tasks)
    lo, hi = max(tasks), sum(tasks)

    while lo < hi:
        mid = (lo + hi) // 2
        if can_distribute(tasks, k, mid):
            hi = mid
        else:
            lo = mid + 1

    print(f"Tasks: {tasks}")
    print(f"Teams: {k}")
    print(f"Minimum possible max load: {lo}")
    return lo

# --- Test ---
tasks = [1, 2, 3, 4, 5, 6, 7, 8]
optimal_partition(tasks, k=3)

---
## 6️⃣ All-in-One Comparison
Run all strategies on the same input and compare results.

In [ ]:
import heapq

tasks = [4, 7, 2, 9, 1, 5, 3]
k = 3

print("=" * 50)
print(f"Tasks: {tasks} | Teams: {k}")
print("=" * 50)

# Strategy 1: Heap-based minimization
heap = [0] * k
heapq.heapify(heap)
for t in sorted(tasks, reverse=True):
    heapq.heappush(heap, heapq.heappop(heap) + t)
print(f"\n📌 Heap Strategy — Max Load: {max(heap)}, Loads: {sorted(heap)}")

# Strategy 2: Round Robin
rr_teams = {i: [] for i in range(k)}
for i, t in enumerate(tasks):
    rr_teams[i % k].append(t)
rr_loads = [sum(v) for v in rr_teams.values()]
print(f"\n🔄 Round Robin  — Max Load: {max(rr_loads)}, Loads: {rr_loads}")

# Strategy 3: Binary Search
def _can(tasks, k, ml):
    need, cur = 1, 0
    for t in tasks:
        if t > ml: return False
        if cur + t > ml: need += 1; cur = t
        else: cur += t
    return need <= k

lo, hi = max(tasks), sum(tasks)
while lo < hi:
    mid = (lo + hi) // 2
    if _can(sorted(tasks), k, mid): hi = mid
    else: lo = mid + 1
print(f"\n🔍 Binary Search — Min possible Max Load: {lo}")

print("\n" + "=" * 50)
print("✅ Best strategy depends on your constraint!")
print("   Use Heap for greedy assignment.")
print("   Use Binary Search for optimal partition.")

---
## 🏆 Challenge Problem — Solve It Yourself!

**Problem:** You have `n` tasks, each with a priority score. Assign them to `k` teams. Each team can handle **at most 3 tasks**. Return the assignment that **maximizes the minimum total priority** any team receives.

```
tasks   = [8, 3, 6, 1, 9, 2, 7, 4, 5]
k       = 3
max_per_team = 3
```

Expected output: each team gets exactly 3 tasks, and the weakest team's total is as high as possible.

💡 Hint: Sort descending, then use heap-based assignment with a cap check.

In [ ]:
# ✏️ Write your solution here!

def challenge_assign(tasks, k, max_per_team):
    # TODO: implement this
    pass

tasks = [8, 3, 6, 1, 9, 2, 7, 4, 5]
result = challenge_assign(tasks, k=3, max_per_team=3)
print(result)

<details>
<summary>💡 Click to reveal solution</summary>

```python
import heapq

def challenge_assign(tasks, k, max_per_team):
    heap = [(0, 0, i) for i in range(k)]  # (load, count, team_index)
    heapq.heapify(heap)
    assignment = {i: [] for i in range(k)}

    for task in sorted(tasks, reverse=True):
        load, count, team = heapq.heappop(heap)
        if count < max_per_team:
            assignment[team].append(task)
            heapq.heappush(heap, (load + task, count + 1, team))
        else:
            heapq.heappush(heap, (load, count, team))  # put back, skip task

    for team, t in assignment.items():
        print(f"Team {team}: {t} -> total={sum(t)}")
    print(f"Min team total: {min(sum(v) for v in assignment.values())}")
    return assignment
```
</details>

---
## 📋 Quick Reference Cheat Sheet

| Pattern | When to Use | Time Complexity |
|---|---|---|
| Min-Heap Greedy | Assign to least loaded team | O(n log k) |
| Round Robin | Equal distribution, no weights | O(n) |
| Earliest Deadline First | Deadline-based scheduling | O(n log n) |
| Binary Search + Greedy | Find optimal max load | O(n log(sum)) |

---
Good luck on your micro1 interview! 🎯